# Pattern quantification
### Mushifugu, Komonfugu

In [ ]:
%load_ext autoreload
%autoreload 1

import os
    
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.stats import ttest_ind, shapiro, levene, probplot

from quant_CV4 import *
%aimport quant_CV4

%matplotlib inline

In [ ]:
df_MK_p = pd.read_csv("../puffer_MK.csv", index_col=0)

### Mushifugu

In [ ]:
df_M = df_MK_p[(df_MK_p["species"]=="mushifugu") & (df_MK_p["image"]==1)]

n = len(df_M)
cols = 5
rows = (n + cols - 1) // cols

fig, axs = plt.subplots(rows, cols, figsize=(12, 12))

axs = axs.flatten()

for i in range(len(axs)):
    ax = axs[i]
    if i < n:
        img = Image.open(os.path.join("mushifugu", df_M.index[i]+".png"))
        ax.imshow(img, cmap='gray')
        ax.set_xlabel(df_M.index[i], fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

plt.tight_layout()
plt.show()


### Komonfugu

In [ ]:
df_K = df_MK_p[(df_MK_p["species"]=="komonfugu") & (df_MK_p["image"]==1)]

n = len(df_K)
cols = 5
rows = (n + cols - 1) // cols

fig, axs = plt.subplots(rows, cols, figsize=(12, 12))

axs = axs.flatten()

for i in range(len(axs)):
    ax = axs[i]
    if i < n:
        img = Image.open(os.path.join("komonfugu", df_K.index[i]+".png"))
        ax.imshow(img, cmap='gray')
        ax.set_xlabel(df_K.index[i], fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

plt.tight_layout()
plt.show()


### Pattern quantification (Mushifugu, Komonfugu)

In [ ]:
for id in df_MK_p[df_MK_p["image"] == 1].index:
    img_path = os.path.join(df_MK_p.loc[id, "species"], f"{id}.png")
    img_gray = Image.open(img_path).convert("L")
    res = quant_CV4.quant(np.asarray(img_gray))
    L, PCS = res[0], res[1]

    df_MK_p.loc[id, "lightness"] = L
    df_MK_p.loc[id, "complexity"] = PCS

# df_MK_p.to_csv("../puffer_MK.csv")
df_MK_p.head()

### RD model (sim_uni)
#### (see Miyazawa 2020 *Science Advances*: [DOI: 10.1126/sciadv.abb9107](https://www.science.org/doi/10.1126/sciadv.abb9107))

In [ ]:
df_uni_p = pd.read_csv("uni_p.csv", index_col=0)
df_uni_p['id'] = ['uni_{:03d}'.format(i) for i in df_uni_p.index]
df_uni_p['species'] = 'sim_uni'
df_uni_p['specimen'] = [fb.split("/")[2] for fb in df_uni_p.file_base]

df_uni = df_uni_p[['id',
                   'species',
                   'lightness',
                   'complexity',
                   'specimen']]

df_uni.head()

### Plotting

In [ ]:
df_MK = df_MK_p[df_MK_p["image"]==1][['species',
                                      'population',
                                      'lightness',
                                      'complexity']]

col_Kj = "khaki"
col_Kp = "orange"
col_Mj = "lightskyblue"
col_Mp = "lightseagreen"

df_MK.loc[(df_MK['species'] == 'komonfugu') & (df_MK['population'] == 'SJ'), 'color'] = col_Kj
df_MK.loc[(df_MK['species'] == 'komonfugu') & (df_MK['population'] == 'SJ'), 'sp_pop'] = 'Kj'

df_MK.loc[(df_MK['species'] == 'komonfugu') & (df_MK['population'] == 'PO'), 'color'] = col_Kp
df_MK.loc[(df_MK['species'] == 'komonfugu') & (df_MK['population'] == 'PO'), 'sp_pop'] = 'Kp'

df_MK.loc[(df_MK['species'] == 'mushifugu') & (df_MK['population'] == 'SJ'), 'color'] = col_Mj
df_MK.loc[(df_MK['species'] == 'mushifugu') & (df_MK['population'] == 'SJ'), 'sp_pop'] = 'Mj'

df_MK.loc[(df_MK['species'] == 'mushifugu') & (df_MK['population'] == 'PO'), 'color'] = col_Mp
df_MK.loc[(df_MK['species'] == 'mushifugu') & (df_MK['population'] == 'PO'), 'sp_pop'] = 'Mp'

df_uni.loc[df_uni['species']=='sim_uni', ['color']] = 'lightgray'

In [ ]:
sns.set(style="ticks", context="notebook")

plt.figure(figsize=(18, 7))
plt_sc = []

numS = 200

plt_sc.append(plt.scatter(df_uni['lightness'],
                          df_uni['complexity'],
                          color=df_uni['color'],
                          s=50,
                          alpha=0.5,
                          label="model"))

order = ["Mj", "Mp", "Kj", "Kp"]

for sp_pop in order:
    group_df = df_MK[df_MK["sp_pop"] == sp_pop]
    plt_sc.append(plt.scatter(group_df['lightness'],
                              group_df['complexity'],
                              color=group_df['color'],
                              marker="o",
                              s=numS,
                              alpha=0.7,
                              label=sp_pop))
            
plt_sc[0].axes.set_xlim(-0.02, 1.05)
plt_sc[0].axes.set_ylim(0.06, 1.02)
plt.xlabel("Overall color tone", fontsize=28)
plt.ylabel("Pattern complexity", fontsize=28)
    
plt.tick_params(axis='both', which='major', labelsize=20)
    
leg = plt.legend(bbox_to_anchor=(1.07, 0.5),
                 loc='center left',
                 ncol=1,
                 frameon=False,
                 fontsize=24)
    
plt.tight_layout()

plt.show()
# plt.savefig("quant_scatter.pdf", transparent = True)


In [ ]:
palette_dict = {
    "Mj": col_Mj,
    "Mp": col_Mp,
    "Kj": col_Kj,
    "Kp": col_Kp
}

In [ ]:
groups = ["Mj", "Mp", "Kj", "Kp"]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

for i, group in enumerate(groups):
    data = df_MK[df_MK['sp_pop'] == group]['complexity'].dropna()
    probplot(data, dist="norm", plot=axes[i])
    axes[i].set_title(f"Q-Q Plot: {group}")

plt.tight_layout()
plt.show()


In [ ]:
groups = ['Mj', 'Mp', 'Kj', 'Kp']
group_data = {g: df_MK[df_MK['sp_pop'] == g]['complexity'].dropna() for g in groups}

# Shapiro-Wilk normality test
print("Shapiro-Wilk test (Normality check)")
for g in groups:
    w, p = shapiro(group_data[g])
    print(f"{g}: W = {w:.3f}, p = {p:.4f} {'→ OK' if p > 0.05 else '→ Non-normal'}")

# Levene's test for homogeneity of variances
print("\nLevene's test (Homogeneity of variance check)")
w, p = levene(*[group_data[g] for g in groups])
print(f"W = {w:.3f}, p = {p:.4f} {'→ OK' if p > 0.05 else '→ Variances unequal'}")

# Student’s t-test (equal_var=True, assumptions met)
print("\nStudent’s t-test (Assumptions met → equal_var=True)")
t1, p_m = ttest_ind(group_data['Mj'], group_data['Mp'], equal_var=True)
print(f"Mj vs Mp: t = {t1:.3f}, p = {p_m:.4f}")

t2, p_k = ttest_ind(group_data['Kj'], group_data['Kp'], equal_var=True)
print(f"Kj vs Kp: t = {t2:.3f}, p = {p_k:.4f}")

# Combine groups for overall comparison
K_all = pd.concat([group_data['Kj'], group_data['Kp']])
M_all = pd.concat([group_data['Mj'], group_data['Mp']])
t3, p_all = ttest_ind(K_all, M_all, equal_var=True)
print(f"K (Kj+Kp) vs M (Mj+Mp): t = {t3:.3f}, p = {p_all:.4f}")


In [ ]:
def p_to_sig(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'n.s.'

def annotate_bracket(x1, x2, y, h, p, ax):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    sig = p_to_sig(p)
    ax.text((x1 + x2) * .5, y + h + 0.005, sig,
            ha='center', va='bottom', fontsize=18)

In [ ]:
plt.figure(figsize=(6, 6))
plt_bx = sns.boxplot(x="sp_pop",
                     y="complexity",
                     hue="sp_pop",
                     data=df_MK,
                     order=["Mj", "Mp", "Kj", "Kp"],
                     linewidth=2.0,
                     palette=palette_dict)

y_max = df_MK['complexity'].max()
gap = 0.025
annotate_bracket(0, 1, y_max + 2*gap, gap, p_m, plt_bx)        # Mj vs Mp
annotate_bracket(2, 3, y_max + gap, gap, p_k, plt_bx)      # Kj vs Kp
annotate_bracket(0, 3, y_max + 5*gap, gap, p_all, plt_bx)    # M vs K

# plt_bx.axes.set_ylim(0.0, 1.05)
plt_bx.set_xlabel("")
plt_bx.set_ylabel("Pattern complexity", fontsize=26)

plt.tick_params(axis='y', which='major', labelsize=16)
plt.tick_params(axis='x', which='major', labelsize=24)
plt.tight_layout()
plt.ylim(0, y_max + 9*gap)

plt.show()


In [ ]:
plt.figure(figsize=(6, 6))
plt_vp=sns.violinplot(x="sp_pop",
                      y="complexity",
                      hue="sp_pop",
                      data=df_MK,
                      order=["Mj", "Mp", "Kj", "Kp"],
                      inner=None,
                      linewidth=0,
                      palette=palette_dict)
plt_sp=sns.swarmplot(x="sp_pop",
                     y="complexity",
                     data=df_MK,
                     size=5,
                     color="black",
                     alpha=0.7)

y_max = df_MK['complexity'].max()
gap = 0.025
annotate_bracket(0, 1, y_max + 4*gap, gap, p_m, plt_vp)        # Mj vs Mp
annotate_bracket(2, 3, y_max + 2*gap, gap, p_k, plt_vp)      # Kj vs Kp
annotate_bracket(0, 3, y_max + 6*gap, gap, p_all, plt_vp)    # M vs K

plt.xlabel("")
plt.ylabel("Pattern complexity", fontsize=26)

plt.tick_params(axis='y', which='major', labelsize=16)
plt.tick_params(axis='x', which='major', labelsize=24)
plt.tight_layout()
plt.ylim(0.1, y_max + 10*gap)

plt.show()